## Fix the broken schakels in N roads
## create shp files for arcgis

In [4]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Input and output paths
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
output_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_edited.gpkg")

# Read the input GeoPackage
gdf_src = gpd.read_file(input_path)

# Define the pairs to merge
pairs = [
    ("033-1030-L", "033-1030-R"),
    ("035-0006-L", "035-0006-R"),
    ("048-1010-L", "048-1010-R"),
    ("035-0002-L", "035-0002-R"),
    ("057-0020-L", "057-0020-R"),
    ("057-0030-L", "057-0030-R"),
    ("057-0050-L", "057-0050-R"),
    ("057-0060-L", "057-0060-R"),
    ("057-0070-L", "057-0070-R"),
    ("057-0080-L", "057-0080-R"),
    ("059-0010-L", "059-0010-R"),
    ("059-0030-L", "059-0030-R"),
    ("059-0060-L", "059-0060-R"),
    ("059-0040-L", "059-0040-R"),
    ("059-0050-L", "059-0050-R"),
    ("059-0020-L", "059-0020-R"),
    ("035-0004-L", "035-0004-R"),
    ("015-0096-L", "015-0096-R"),

]

# Columns to sum
sum_cols = ["total_length", "flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]

# Column to take max
max_col = "F_EV2_ma_max"

merged_rows = []
merged_ids = set()

for left, right in pairs:
    row_L = gdf_src[gdf_src["NETWERKSCH_HWN"] == left]
    row_R = gdf_src[gdf_src["NETWERKSCH_HWN"] == right]

    if row_L.empty or row_R.empty:
        continue

    row_L = row_L.iloc[0]
    row_R = row_R.iloc[0]

    merged_data = row_R.copy()

    # Sum numeric columns
    for col in sum_cols:
        merged_data[col] = row_L[col] + row_R[col]

    # Max for F_EV2_ma_max
    merged_data[max_col] = max(row_L[max_col], row_R[max_col])

    # Compute fraction_flooded
    merged_data["fraction_flooded"] = (
        merged_data["flooded_length"] / merged_data["total_length"]
        if merged_data["total_length"] != 0 else 0
    )

    merged_rows.append(merged_data)

    # Track merged IDs
    merged_ids.update([left, right])

# Create GeoDataFrame for merged rows
gdf_merged_pairs = gpd.GeoDataFrame(merged_rows, crs=gdf_src.crs)

# Keep all other rows that were not merged
gdf_rest = gdf_src[~gdf_src["NETWERKSCH_HWN"].isin(merged_ids)]

# Combine merged pairs with the rest
gdf_final = gpd.GeoDataFrame(pd.concat([gdf_merged_pairs, gdf_rest], ignore_index=True), crs=gdf_src.crs)

# Save to GeoPackage
gdf_final.to_file(output_path, driver="GPKG")

print(f"Final dataset saved with {len(gdf_final)} rows to {output_path}")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_edited')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_edited')) failed: unable to open database file"


Final dataset saved with 468 rows to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_edited.gpkg


In [ ]:
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
gdf_area = gpd.read_file(area_gpkg)

AFR_path  = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\AFR_Points.shp")
OPR_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\OPR_Points.shp")

OPR = gpd.read_file(OPR_path)
AFR = gpd.read_file(AFR_path)



In [2]:
import geopandas as gpd
from pathlib import Path

# Load data
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
gdf_area = gpd.read_file(area_gpkg)

AFR_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\AFR_Points.shp")
OPR_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\OPR_Points.shp")

OPR = gpd.read_file(OPR_path)
AFR = gpd.read_file(AFR_path)

# Check the column name for area name in gdf_area
print(gdf_area.columns)  # Look for something like 'area_name'

# Perform spatial join
OPR_with_area = gpd.sjoin(OPR, gdf_area[['geometry', 'name']], how='left', predicate='within')
AFR_with_area = gpd.sjoin(AFR, gdf_area[['geometry', 'name']], how='left', predicate='within')

# Save updated files
OPR_with_area.to_file(OPR_path, driver='ESRI Shapefile')
AFR_with_area.to_file(AFR_path, driver='ESRI Shapefile')

Index(['gml_id', 'nationalCo', 'localId', 'namespace', 'nationalLe',
       'national_1', 'country', 'name', 'geometry'],
      dtype='object')


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_11916\535882985.py:22: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  OPR_with_area.to_file(OPR_path, driver='ESRI Shapefile')
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_11916\535882985.py:23: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  AFR_with_area.to_file(AFR_path, driver='ESRI Shapefile')


In [5]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Input and output paths
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
output_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver02.gpkg")

# Read the input GeoPackage
gdf_src = gpd.read_file(input_path)

# Columns to sum
sum_cols = ["total_length", "flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]

# Column to take max
max_col = "F_EV2_ma_max"

# Create aggregation dictionary
agg_dict = {col: 'sum' for col in sum_cols}
agg_dict[max_col] = 'max'

# Dissolve based on NETWERKSCH_HWN column
gdf_dissolved = gdf_src.dissolve(by='NET', aggfunc=agg_dict)

# Reset index to make NETWERKSCH_HWN a column again
gdf_dissolved = gdf_dissolved.reset_index()

# Calculate fraction_flooded and dam_p_m
gdf_dissolved["fraction_flooded"] = (
    gdf_dissolved["flooded_length"] / gdf_dissolved["total_length"]
).fillna(0)

gdf_dissolved["dam_p_m"] = (
    gdf_dissolved["total_damage"] / gdf_dissolved["total_length"]
).fillna(0)

display (gdf_dissolved.head())

# Save to GeoPackage
gdf_dissolved.to_file(output_path, driver="GPKG")

print(f"Dissolved dataset saved with {len(gdf_dissolved)} rows to {output_path}")


,NET,geometry,total_length,flooded_length,bridge_length_sum,tunnel_length_sum,total_damage,F_EV2_ma_max,fraction_flooded,dam_p_m
0,001-0010,"MULTILINESTRING ((127717.503 483555.369, 12770...",13152.0,295.428930,254.201591,0.000000,110007.0,698.000000,0.022463,8.364279
1,001-0020,"MULTILINESTRING ((134398.600 481503.333, 13436...",35878.0,0.000000,937.758472,1421.200037,0.0,0.000000,0.000000,0.000000
2,001-0030,"MULTILINESTRING ((144278.984 472187.502, 14425...",37070.0,10.000000,770.275529,0.000000,403.0,52.000000,0.000270,0.010871
3,001-0040,"MULTILINESTRING ((158887.014 464473.984, 15887...",59163.0,956.884374,451.959035,0.000000,42858.0,187.416794,0.016174,0.724405
4,001-0055,"MULTILINESTRING ((161365.083 464125.423, 16135...",5214.0,0.000000,50.847451,0.000000,0.0,0.000000,0.000000,0.000000


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver02')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver02')) failed: unable to open database file"


Dissolved dataset saved with 244 rows to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver02.gpkg


In [7]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Input and output paths
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
output_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver04.gpkg")

# Read the input GeoPackage
gdf_src = gpd.read_file(input_path)

# Columns to sum
sum_cols = ["total_length", "flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]

# Column to take max
max_col = "F_EV2_ma_max"

# First, dissolve geometry by NET
geom_dissolved = gdf_src.dissolve(by='NET')[['geometry']]

# Create aggregation dictionary for non-geometry columns
agg_dict = {col: 'sum' for col in sum_cols}
agg_dict[max_col] = 'max'

# Group and aggregate the data columns
data_aggregated = gdf_src.groupby('NET')[sum_cols + [max_col]].agg(agg_dict).reset_index()

# Merge the aggregated data with dissolved geometry
gdf_dissolved = data_aggregated.merge(geom_dissolved, left_on='NET', right_index=True)
gdf_dissolved = gpd.GeoDataFrame(gdf_dissolved, geometry='geometry', crs=gdf_src.crs)

# Calculate fraction_flooded and dam_p_m
gdf_dissolved["fraction_flooded"] = (
    gdf_dissolved["flooded_length"] / gdf_dissolved["total_length"]
).fillna(0)

gdf_dissolved["dam_p_m"] = (
    gdf_dissolved["total_damage"] / gdf_dissolved["total_length"]
).fillna(0)

display(gdf_dissolved.head())
print(f"Total rows before dissolve: {len(gdf_src)}")
print(f"Total rows after dissolve: {len(gdf_dissolved)}")

# Debug: Check a sample NET group to verify summing
sample_net = gdf_src['NET'].iloc[0]
print(f"\nDebug check for NET={sample_net}:")
sample_group = gdf_src[gdf_src['NET'] == sample_net]
print(f"  Original rows: {len(sample_group)}")
print(f"  Sum of total_damage: {sample_group['total_damage'].sum()}")
dissolved_value = gdf_dissolved[gdf_dissolved['NET'] == sample_net]['total_damage'].values[0]
print(f"  Dissolved total_damage: {dissolved_value}")

# Save to GeoPackage
gdf_dissolved.to_file(output_path, driver="GPKG")

print(f"\nDissolved dataset saved with {len(gdf_dissolved)} rows to {output_path}")

,NET,total_length,flooded_length,bridge_length_sum,tunnel_length_sum,total_damage,F_EV2_ma_max,geometry,fraction_flooded,dam_p_m
0,001-0010,13152.0,295.428930,254.201591,0.000000,110007.0,698.000000,"MULTILINESTRING ((127717.503 483555.369, 12770...",0.022463,8.364279
1,001-0020,35878.0,0.000000,937.758472,1421.200037,0.0,0.000000,"MULTILINESTRING ((134398.600 481503.333, 13436...",0.000000,0.000000
2,001-0030,37070.0,10.000000,770.275529,0.000000,403.0,52.000000,"MULTILINESTRING ((144278.984 472187.502, 14425...",0.000270,0.010871
3,001-0040,59163.0,956.884374,451.959035,0.000000,42858.0,187.416794,"MULTILINESTRING ((158887.014 464473.984, 15887...",0.016174,0.724405
4,001-0055,5214.0,0.000000,50.847451,0.000000,0.0,0.000000,"MULTILINESTRING ((161365.083 464125.423, 16135...",0.000000,0.000000


Total rows before dissolve: 486
Total rows after dissolve: 244

Debug check for NET=001-0010:
  Original rows: 2
  Sum of total_damage: 110007.0
  Dissolved total_damage: 110007.0


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver04')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver04')) failed: unable to open database file"



Dissolved dataset saved with 244 rows to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver04.gpkg


In [6]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Input and output paths
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
output_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver03.gpkg")

# Read the input GeoPackage
gdf_src = gpd.read_file(input_path)

# Columns to sum
sum_cols = ["total_length", "flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]

# Column to take max
max_col = "F_EV2_ma_max"

# Create aggregation dictionary
agg_dict = {col: 'sum' for col in sum_cols}
agg_dict[max_col] = 'max'

# Dissolve based on NET column with proper aggregation
gdf_dissolved = gdf_src.dissolve(by='NET', aggfunc=agg_dict).reset_index()

# Alternative approach if above doesn't work:
# gdf_dissolved = gdf_src.groupby('NET').agg(agg_dict).reset_index()
# Then merge back the geometry:
# gdf_dissolved = gdf_dissolved.merge(gdf_src.dissolve(by='NET')[['geometry']], left_on='NET', right_index=True)
# gdf_dissolved = gpd.GeoDataFrame(gdf_dissolved, geometry='geometry', crs=gdf_src.crs)

# Calculate fraction_flooded and dam_p_m
gdf_dissolved["fraction_flooded"] = (
    gdf_dissolved["flooded_length"] / gdf_dissolved["total_length"]
).fillna(0)

gdf_dissolved["dam_p_m"] = (
    gdf_dissolved["total_damage"] / gdf_dissolved["total_length"]
).fillna(0)

display(gdf_dissolved.head())
print(f"Total rows before dissolve: {len(gdf_src)}")
print(f"Total rows after dissolve: {len(gdf_dissolved)}")

# Save to GeoPackage
gdf_dissolved.to_file(output_path, driver="GPKG")

print(f"Dissolved dataset saved with {len(gdf_dissolved)} rows to {output_path}")

,NET,geometry,total_length,flooded_length,bridge_length_sum,tunnel_length_sum,total_damage,F_EV2_ma_max,fraction_flooded,dam_p_m
0,001-0010,"MULTILINESTRING ((127717.503 483555.369, 12770...",13152.0,295.428930,254.201591,0.000000,110007.0,698.000000,0.022463,8.364279
1,001-0020,"MULTILINESTRING ((134398.600 481503.333, 13436...",35878.0,0.000000,937.758472,1421.200037,0.0,0.000000,0.000000,0.000000
2,001-0030,"MULTILINESTRING ((144278.984 472187.502, 14425...",37070.0,10.000000,770.275529,0.000000,403.0,52.000000,0.000270,0.010871
3,001-0040,"MULTILINESTRING ((158887.014 464473.984, 15887...",59163.0,956.884374,451.959035,0.000000,42858.0,187.416794,0.016174,0.724405
4,001-0055,"MULTILINESTRING ((161365.083 464125.423, 16135...",5214.0,0.000000,50.847451,0.000000,0.0,0.000000,0.000000,0.000000


Total rows before dissolve: 486
Total rows after dissolve: 244


CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: unable to open database file'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: unable to open database file'


Dissolved dataset saved with 244 rows to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_wholeSchakel_ver03.gpkg
